# LeetCode 289 — Game of Life bằng Python

Link đề: https://leetcode.com/problems/game-of-life/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Simulation

Skeleton LeetCode:

```python
class Solution:
    def gameOfLife(self, board: List[List[int]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (đếm hàng xóm trên bảng có viền đệm).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mô phỏng **một thế hệ** của **Trò chơi Cuộc sống (Conway's Game of Life)** trên lưới `m × n`: mỗi ô **sống (`1`)** hoặc **chết (`0`)**, có **8 hàng xóm** (ngang, dọc, chéo), và **bốn luật** áp dụng **đồng thời** cho mọi ô.
2. Hiểu vì sao phải cập nhật **đồng thời**: nếu sửa ô rồi dùng ngay giá trị mới để đếm hàng xóm của ô kế tiếp thì kết quả **sai**.
3. Đi từ **sao chép bảng** `O(mn)` bộ nhớ → **mã hóa trạng thái trong chính bảng** (`2`: sống→chết, `3`: chết→sống) `O(1)` bộ nhớ → **mã hóa bằng bit** (bit 0 = hiện tại, bit 1 = kế tiếp) → **tập các ô sống** để giải **bảng vô hạn** (follow-up 2).
4. Làm quen với **duyệt 8 hướng**, kiểm tra biên và **bảng đệm**; quen với kỹ thuật **trạng thái trung gian** để sửa tại chỗ.
5. Tránh các lỗi kinh điển: cập nhật không đồng thời, đếm **cả ô đang xét**, đọc trạng thái đã mã hóa như số nguyên (`2`, `3`), sai biên, và quên bước **giải mã** cuối cùng.

## 2. Đề bài & ví dụ

Theo Wikipedia, **Trò chơi Cuộc sống (Life)** là một **automata tế bào** do nhà toán học John Horton Conway đặt ra năm 1970.

Bảng gồm lưới `m × n` ô, mỗi ô có trạng thái ban đầu: **sống (`1`)** hoặc **chết (`0`)**. Mỗi ô tương tác với **tám hàng xóm** (ngang, dọc, chéo) theo bốn luật:

1. Ô **sống** có **ít hơn hai** hàng xóm sống thì **chết** (do thiếu dân số).
2. Ô **sống** có **hai hoặc ba** hàng xóm sống thì **sống tiếp** sang thế hệ sau.
3. Ô **sống** có **nhiều hơn ba** hàng xóm sống thì **chết** (do quá đông).
4. Ô **chết** có **đúng ba** hàng xóm sống thì **sống lại** (sinh sản).

Trạng thái kế tiếp được xác định bằng cách áp dụng các luật trên **đồng thời** cho mọi ô của trạng thái hiện tại. Cho bảng hiện tại, hãy **cập nhật bảng thành trạng thái kế tiếp** (không cần trả về gì).

**Ví dụ 1**

- Input: `board = [[0,1,0],[0,0,1],[1,1,1],[0,0,0]]`
- Output: `[[0,0,0],[1,0,1],[0,1,1],[0,1,0]]`

**Ví dụ 2**

- Input: `board = [[1,1],[1,0]]`
- Output: `[[1,1],[1,1]]`

**Ràng buộc:** `m == board.length`; `n == board[i].length`; `1 <= m, n <= 25`; `board[i][j]` là `0` hoặc `1`.

**Follow up:** (a) giải **tại chỗ (in-place)**? Nhớ cập nhật **đồng thời**; (b) bảng thực ra **vô hạn**: khi ô sống chạm biên thì sao?

In [1]:
from typing import List, Set, Tuple
from collections import Counter
import copy
import random
import time

examples = [
    ([[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]], [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]),
    ([[1, 1], [1, 0]], [[1, 1], [1, 1]]),
]


def show(board):
    return "\n".join("  " + " ".join(".#"[v & 1] if v in (0, 1) else str(v) for v in row) for row in board)


print(show(examples[0][0]))

  . # .
  . . #
  # # #
  . . .


## 3. Kiến thức nền

### 3.1. Tám hàng xóm và kiểm tra biên

Ô `(r, c)` có hàng xóm `(r+dr, c+dc)` với `dr, dc ∈ {-1, 0, 1}` trừ `(0, 0)`. Ô ở biên có **ít hơn 8** hàng xóm: phải kiểm tra `0 <= nr < m` và `0 <= nc < n` (hàng xóm ngoài bảng coi như **chết**).
#### Code demo

In [2]:
NEIGHBORS = [(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]
print(NEIGHBORS)                     # 8 hướng


def count_live(board, r, c):
    m, n = len(board), len(board[0])
    total = 0
    for dr, dc in NEIGHBORS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            total += board[nr][nc]
    return total


b = examples[0][0]
print(count_live(b, 0, 0), count_live(b, 2, 1), count_live(b, 3, 1))     # 0 hàng xóm sống ở góc, v.v.

[(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
1 3 3


### 3.2. Bốn luật rút gọn thành hai

Gọi `k` là số hàng xóm sống. Ô **sống ở thế hệ sau** khi: `k == 3` (bất kể trước đó), hoặc (`ô đang sống` và `k == 2`). Ngược lại chết. Kiểm tra: ô sống với `k = 2, 3` sống tiếp (luật 2), ô chết với `k = 3` sinh sản (luật 4), còn lại chết (luật 1, 3).
#### Code demo

In [3]:
def next_state(alive, k):
    return 1 if (k == 3 or (alive == 1 and k == 2)) else 0


for alive in (0, 1):
    print(f"ô {'sống' if alive else 'chết'}:", {k: next_state(alive, k) for k in range(9)})

ô chết: {0: 0, 1: 0, 2: 0, 3: 1, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0}
ô sống: {0: 0, 1: 0, 2: 1, 3: 1, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0}


### 3.3. Cập nhật đồng thời và vì sao sửa tại chỗ ngây thơ là sai

Nếu ta vừa duyệt vừa ghi `board[r][c] = next_state(...)` thì các ô duyệt sau sẽ đếm hàng xóm dựa trên giá trị **đã thay đổi** (thuộc thế hệ **sau**) của các ô duyệt trước ⇒ kết quả **khác** đáp án đúng. Phải tách việc **đọc trạng thái cũ** khỏi việc **ghi trạng thái mới**.
#### Code demo

In [4]:
def naive_wrong(board):
    m, n = len(board), len(board[0])
    for r in range(m):
        for c in range(n):
            board[r][c] = next_state(board[r][c], count_live(board, r, c))     # SAI: đếm trên bảng đang bị sửa


b = copy.deepcopy(examples[0][0])
naive_wrong(b)
print("kết quả ngây thơ:", b, "\n| đúng            :", examples[0][1])

kết quả ngây thơ: [[0, 0, 0], [0, 0, 1], [0, 1, 1], [0, 0, 0]] 
| đúng            : [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]


### 3.4. Mã hóa nhiều trạng thái trong một ô

Ta có thể lưu **cả trạng thái cũ lẫn mới** trong cùng một ô bằng cách dùng thêm giá trị: `0` (chết→chết), `1` (sống→sống), `2` (sống→chết), `3` (chết→sống). Trạng thái **cũ** của ô là: `old = 1 nếu giá trị ∈ {1, 2} ngược lại 0`, tức `value in (1, 2)`; hoặc dùng bit: **bit 0 = cũ**, **bit 1 = mới**. Khi đếm hàng xóm ta chỉ đọc bit cũ (`value & 1`) nên không bị ảnh hưởng bởi trạng thái mới đã ghi.
#### Code demo

In [5]:
for v, meaning in [(0, "chết -> chết"), (1, "sống -> sống"), (2, "sống -> chết"), (3, "chết -> sống")]:
    old_by_membership = 1 if v in (1, 2) else 0       # cách 1: tra thành viên
    print(f"giá trị {v}: {meaning}; trạng thái cũ = {old_by_membership}")
# cách bit (quy ước khác): bit0 = trạng thái cũ, bit1 = trạng thái mới
for old in (0, 1):
    for new in (0, 1):
        v = old | (new << 1)
        print(f"old={old}, new={new} -> v={v}, old = v & 1 = {v & 1}, new = v >> 1 = {v >> 1}")

giá trị 0: chết -> chết; trạng thái cũ = 0
giá trị 1: sống -> sống; trạng thái cũ = 1
giá trị 2: sống -> chết; trạng thái cũ = 1
giá trị 3: chết -> sống; trạng thái cũ = 0
old=0, new=0 -> v=0, old = v & 1 = 0, new = v >> 1 = 0
old=0, new=1 -> v=2, old = v & 1 = 0, new = v >> 1 = 1
old=1, new=0 -> v=1, old = v & 1 = 1, new = v >> 1 = 0
old=1, new=1 -> v=3, old = v & 1 = 1, new = v >> 1 = 1


### 3.5. Bảng vô hạn và tập các ô sống

Với bảng **vô hạn**, ta không lưu lưới mà lưu **tập các tọa độ ô sống**. Để tính thế hệ sau: với mỗi ô sống, **cộng `1` vào bộ đếm** của cả 8 hàng xóm (`Counter`); rồi ô sống ở thế hệ sau là ô có bộ đếm `== 3`, hoặc bộ đếm `== 2` và đang sống. Công việc tỉ lệ với **số ô sống**, không phụ thuộc kích thước bảng.
#### Code demo

In [6]:
live = {(0, 1), (1, 2), (2, 0), (2, 1), (2, 2)}      # glider
counts = Counter((r + dr, c + dc) for r, c in live for dr, dc in NEIGHBORS)
nxt = {cell for cell, k in counts.items() if k == 3 or (k == 2 and cell in live)}
print(sorted(nxt))

[(1, 0), (1, 2), (2, 1), (2, 2), (3, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 2: `[[1,1],[1,0]]`:

- Ô `(0,0)` sống, hàng xóm sống: `(0,1)`, `(1,0)` ⇒ `k = 2` ⇒ sống tiếp.
- Ô `(0,1)` sống, hàng xóm sống: `(0,0)`, `(1,0)` ⇒ `k = 2` ⇒ sống tiếp.
- Ô `(1,0)` sống, hàng xóm sống: `(0,0)`, `(0,1)` ⇒ `k = 2` ⇒ sống tiếp.
- Ô `(1,1)` chết, hàng xóm sống: cả ba ô còn lại ⇒ `k = 3` ⇒ **sinh sản**.

Kết quả `[[1,1],[1,1]]`. **Cách nghĩ:** với mỗi ô, nhìn **bản chụp cũ** để đếm hàng xóm sống rồi áp luật; việc ghi phải xảy ra trên một chỗ khác, hoặc ghi theo cách **không phá** thông tin cũ.

## 5. Cách 1 — Brute Force / baseline: sao chép bảng

**Ý tưởng.** Tạo bản sao `old` của `board`; với mỗi ô, đếm hàng xóm sống **trên `old`** rồi ghi trạng thái mới vào `board`. Đơn giản và chắc chắn đúng.
#### Code demo

In [7]:
def game_of_life_copy(board):
    m, n = len(board), len(board[0])
    old = [row[:] for row in board]                    # bản sao: trạng thái cũ
    for r in range(m):
        for c in range(n):
            k = 0
            for dr, dc in NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    k += old[nr][nc]
            board[r][c] = 1 if (k == 3 or (old[r][c] == 1 and k == 2)) else 0


for start, expected in examples:
    b = copy.deepcopy(start)
    game_of_life_copy(b)
    print("khớp kỳ vọng:", b == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


**Độ phức tạp (Cách 1).** Gọi `N = m·n`: **Time `O(N)`** (mỗi ô xét 8 hàng xóm), **Space `O(N)`** (bản sao). Thỏa bài nhưng chưa thỏa follow-up (a).

**Nút thắt.** Ta tốn thêm `O(mn)` bộ nhớ chỉ để nhớ trạng thái cũ. Có thể nhớ nó **ngay trong bảng** nhờ mã hóa.

## 6. Ý tưởng thuật toán / bước đột phá

**Mã hóa hai trạng thái trong một ô.** Dùng các giá trị:

- `0`: chết → chết; `1`: sống → sống;
- `2`: **sống → chết**; `3`: **chết → sống**.

Trạng thái **cũ** của ô: sống nếu giá trị `∈ {1, 2}`. Duyệt **lượt 1**: với mỗi ô đếm hàng xóm sống **theo trạng thái cũ** (`value in (1, 2)`) rồi ghi `2` hoặc `3` nếu cần thay đổi (giữ nguyên `0`/`1` nếu không đổi). Duyệt **lượt 2**: **giải mã**: `2 → 0`, `3 → 1`.

Cách này dùng **`O(1)`** bộ nhớ phụ và cập nhật **đồng thời** đúng vì mọi lần đếm đều chỉ nhìn **trạng thái cũ** được suy ra từ giá trị mã hóa.

## 7. Các cách tiếp theo

### Cách 2 — Tại chỗ với mã hóa `2` / `3` (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [8]:
def game_of_life_inplace_states(board):
    m, n = len(board), len(board[0])
    for r in range(m):                                  # lượt 1: tính, ghi trạng thái trung gian
        for c in range(n):
            k = 0
            for dr, dc in NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and board[nr][nc] in (1, 2):     # trạng thái CŨ là sống: 1 hoặc 2
                    k += 1
            if board[r][c] == 1 and (k < 2 or k > 3):
                board[r][c] = 2                         # sống -> chết
            elif board[r][c] == 0 and k == 3:
                board[r][c] = 3                         # chết -> sống
    for r in range(m):                                  # lượt 2: giải mã
        for c in range(n):
            if board[r][c] == 2:
                board[r][c] = 0
            elif board[r][c] == 3:
                board[r][c] = 1


for start, expected in examples:
    b = copy.deepcopy(start)
    game_of_life_inplace_states(b)
    print("khớp kỳ vọng:", b == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def gameOfLife(self, board: List[List[int]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
        m, n = len(board), len(board[0])
        dirs = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
        for i in range(m):
            for j in range(n):
                live = 0
                for a, b in dirs:
                    x, y = i + a, j + b
                    if 0 <= x < m and 0 <= y < n and board[x][y] in (1, 2):
                        live += 1
                if board[i][j] == 1 and (live < 2 or live > 3):
                    board[i][j] = 2
                if board[i][j] == 0 and live == 3:
                    board[i][j] = 3
        for i in range(m):
            for j in range(n):
                board[i][j] %= 2


b = copy.deepcopy(examples[0][0])
Solution().gameOfLife(b)
print(b == examples[0][1])

True


**Độ phức tạp Cách 2:** **Time `O(mn)`**, **Space `O(1)`** phụ (tại chỗ). Lưu ý dòng cuối `board[i][j] %= 2`: `2 → 0`, `3 → 1`, `0 → 0`, `1 → 1`.

### Cách 3 — Tại chỗ với mã hóa bit (bit 0 = cũ, bit 1 = mới)

**Giải thích.** Giữ trạng thái cũ ở bit `0`. Tính trạng thái mới `new` rồi ghi vào bit `1`: `board[r][c] |= new << 1`. Khi đếm hàng xóm chỉ đọc `board[nr][nc] & 1`. Cuối cùng `board[r][c] >>= 1`.
#### Code demo

In [10]:
def game_of_life_bits(board):
    m, n = len(board), len(board[0])
    for r in range(m):
        for c in range(n):
            k = 0
            for dr, dc in NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    k += board[nr][nc] & 1                 # chỉ đọc bit cũ
            old = board[r][c] & 1
            new = 1 if (k == 3 or (old == 1 and k == 2)) else 0
            board[r][c] |= new << 1                        # ghi bit mới
    for r in range(m):
        for c in range(n):
            board[r][c] >>= 1                              # giải mã: bỏ bit cũ, giữ bit mới


for start, expected in examples:
    b = copy.deepcopy(start)
    game_of_life_bits(b)
    print("khớp kỳ vọng:", b == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


**Độ phức tạp Cách 3:** **Time `O(mn)`**, **Space `O(1)`** phụ. Cách bit gọn và dễ giải thích; mã hóa `2/3` gần với cách nghĩ "chuyển trạng thái".

### Cách 4 — Tập các ô sống (bảng vô hạn, follow-up 2)

**Giải thích.** Lưu **tập `live`** các tọa độ ô sống. Mỗi thế hệ: `Counter` đếm số hàng xóm sống cho mọi ô nằm cạnh ô sống; ô sống ở thế hệ sau là ô có `k == 3` hoặc (`k == 2` và đang sống). Không có "biên": bảng thực sự vô hạn. Để áp vào bài (bảng hữu hạn), chỉ giữ các ô trong biên sau mỗi bước (hoặc xem bảng hữu hạn là phần cửa sổ của bảng vô hạn **với ngoài cửa sổ luôn chết**).
#### Code demo

In [11]:
def step_live_set(live):
    counts = Counter((r + dr, c + dc) for r, c in live for dr, dc in NEIGHBORS)
    return {cell for cell, k in counts.items() if k == 3 or (k == 2 and cell in live)}


def game_of_life_sparse(board):
    """Dùng cho bảng hữu hạn: tính trên tập ô sống rồi lọc theo biên (ngoài biên luôn chết)."""
    m, n = len(board), len(board[0])
    live = {(r, c) for r in range(m) for c in range(n) if board[r][c] == 1}
    nxt = {(r, c) for (r, c) in step_live_set(live) if 0 <= r < m and 0 <= c < n}
    for r in range(m):
        for c in range(n):
            board[r][c] = 1 if (r, c) in nxt else 0


for start, expected in examples:
    b = copy.deepcopy(start)
    game_of_life_sparse(b)
    print("khớp kỳ vọng:", b == expected)

khớp kỳ vọng: True
khớp kỳ vọng: True


**Độ phức tạp Cách 4:** gọi `L` là số ô sống: **Time `O(L)`** mỗi thế hệ (mỗi ô sống cộng 8 hàng xóm vào `Counter`), **Space `O(L)`**. Hợp với bảng thưa hoặc vô hạn; với bảng đặc (`L ≈ mn`) thì không nhanh hơn Cách 2.

> Cách 2 là bản nộp phổ biến; Cách 3 là biến thể bit; Cách 4 trả lời follow-up (b).

## 8. TRACE từng bước

Ta in bảng sau lượt 1 (có các giá trị `2` và `3`) và sau lượt 2 (giải mã) cho ví dụ 1, kèm số hàng xóm sống của từng ô.
#### Code demo

In [12]:
def trace_states(board):
    m, n = len(board), len(board[0])
    print("bảng ban đầu:")
    print(show(board))
    counts = [[0] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            counts[r][c] = sum(1 for dr, dc in NEIGHBORS
                               if 0 <= r + dr < m and 0 <= c + dc < n and board[r + dr][c + dc] in (1, 2))
            if board[r][c] == 1 and (counts[r][c] < 2 or counts[r][c] > 3):
                board[r][c] = 2
            elif board[r][c] == 0 and counts[r][c] == 3:
                board[r][c] = 3
    print("\nsố hàng xóm sống (tính trên trạng thái cũ):")
    print("\n".join("  " + " ".join(map(str, row)) for row in counts))
    print("\nsau lượt 1 (2 = sống->chết, 3 = chết->sống):")
    print("\n".join("  " + " ".join(map(str, row)) for row in board))
    for r in range(m):
        for c in range(n):
            board[r][c] %= 2
    print("\nsau lượt 2 (giải mã):")
    print("\n".join("  " + " ".join(map(str, row)) for row in board))
    return board


trace_states(copy.deepcopy(examples[0][0]))

bảng ban đầu:
  . # .
  . . #
  # # #
  . . .

số hàng xóm sống (tính trên trạng thái cũ):
  1 1 2
  3 5 3
  1 3 2
  2 3 2

sau lượt 1 (2 = sống->chết, 3 = chết->sống):
  0 2 0
  3 0 1
  2 1 1
  0 3 0

sau lượt 2 (giải mã):
  0 0 0
  1 0 1
  0 1 1
  0 1 0


[[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]]

Bảng tóm tắt (ví dụ 2 `[[1,1],[1,0]]`):

| ô | trạng thái cũ | hàng xóm sống | luật | mã sau lượt 1 | kết quả |
|---|---|---|---|---|---|
| `(0,0)` | sống | 2 | sống tiếp | `1` | `1` |
| `(0,1)` | sống | 2 | sống tiếp | `1` | `1` |
| `(1,0)` | sống | 2 | sống tiếp | `1` | `1` |
| `(1,1)` | chết | 3 | sinh sản | `3` | `1` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (mã hóa `2`/`3`). Đúng `class Solution`, đúng `gameOfLife(self, board: List[List[int]]) -> None`, sửa `board` **tại chỗ**.

In [13]:
class Solution:
    def gameOfLife(self, board: List[List[int]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
        m, n = len(board), len(board[0])
        dirs = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
        for i in range(m):
            for j in range(n):
                live = 0
                for a, b in dirs:
                    x, y = i + a, j + b
                    if 0 <= x < m and 0 <= y < n and board[x][y] in (1, 2):     # trạng thái cũ sống: 1 hoặc 2
                        live += 1
                if board[i][j] == 1 and (live < 2 or live > 3):
                    board[i][j] = 2                                             # sống -> chết
                elif board[i][j] == 0 and live == 3:
                    board[i][j] = 3                                             # chết -> sống
        for i in range(m):
            for j in range(n):
                board[i][j] %= 2                                                # giải mã: 2 -> 0, 3 -> 1


sol = Solution()
for start, expected in examples:
    b = copy.deepcopy(start)
    sol.gameOfLife(b)
    assert b == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Cập nhật đồng thời đúng:** ở lượt 1, khi xét ô `(r, c)`, mọi ô đã xử lý trước nó mang giá trị `0..3` mà **trạng thái cũ** của chúng luôn đọc được: `1` hoặc `2` là "cũ sống", `0` hoặc `3` là "cũ chết". Ta không bao giờ đọc trạng thái mới khi đếm; vì vậy số hàng xóm sống tính được **đúng bằng** số tính trên bảng gốc.
- **Mã hóa đủ thông tin:** bốn giá trị tương ứng đủ bốn cặp (cũ, mới) ∈ {0,1}²: `0 = (0,0)`, `1 = (1,1)`, `2 = (1,0)`, `3 = (0,1)`. Ô giữ nguyên trạng thái (sống–sống hoặc chết–chết) không cần ghi lại.
- **Luật:** ô sống có `live < 2` hoặc `> 3` chết (`1 → 2`); ô chết có `live == 3` sống (`0 → 3`); các trường hợp còn lại giữ nguyên — đúng bốn luật.
- **Giải mã:** `% 2` cho `0→0, 1→1, 2→0, 3→1`, tức lấy trạng thái **mới**.
- **Cách 3 (bit):** bit 0 giữ trạng thái cũ cho tới khi `>>= 1` bỏ nó đi; bit 1 giữ trạng thái mới.
- **Cách 4:** mỗi ô có `k` hàng xóm sống thì có đúng `k` ô sống "đóng góp" `+1` vào bộ đếm của nó; ô không nằm cạnh ô sống nào có `k = 0` nên không bao giờ sống (đúng luật).

## 11. Phân tích Time / Space Complexity

Gọi `N = m·n` (tối đa `625`), `L` số ô sống.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Sao chép bảng | `O(N)` | `O(N)` |
| 2. Mã hóa `2`/`3` | `O(N)` | `O(1)` |
| 3. Mã hóa bit | `O(N)` | `O(1)` |
| 4. Tập ô sống | `O(L)` mỗi thế hệ | `O(L)` |

Mỗi ô xét tối đa `8` hàng xóm ⇒ hệ số hằng `8`. Với `N ≤ 625` mọi cách tức thời. Cách 2/3 là tối ưu về bộ nhớ cho bảng hữu hạn đặc; Cách 4 tối ưu cho bảng thưa hoặc vô hạn. Không có khác biệt average/worst.

## 12. Bộ test

`oracle` là brute force độc lập: dựng **bảng đệm** có viền `0` quanh bảng (kích thước `(m+2) × (n+2)`), đếm hàng xóm bằng **tổng khối `3 × 3` trừ ô giữa** — không dùng danh sách `NEIGHBORS`, không kiểm tra biên.
#### Code demo

In [14]:
def oracle(board):
    m, n = len(board), len(board[0])
    padded = [[0] * (n + 2) for _ in range(m + 2)]
    for r in range(m):
        for c in range(n):
            padded[r + 1][c + 1] = board[r][c]
    result = [[0] * n for _ in range(m)]
    for r in range(1, m + 1):
        for c in range(1, n + 1):
            block = sum(padded[r + i][c + j] for i in (-1, 0, 1) for j in (-1, 0, 1))
            k = block - padded[r][c]                       # tổng khối 3x3 trừ chính ô giữa
            if padded[r][c] == 1:
                result[r - 1][c - 1] = 1 if k in (2, 3) else 0
            else:
                result[r - 1][c - 1] = 1 if k == 3 else 0
    return result


def run_inplace(fn, board):
    b = copy.deepcopy(board)
    fn(b)
    return b


impls = {
    "copy": game_of_life_copy,
    "states": game_of_life_inplace_states,
    "bits": game_of_life_bits,
    "sparse": game_of_life_sparse,
    "submit": sol.gameOfLife,
}

for start, expected in examples:
    assert oracle(start) == expected
    for key, fn in impls.items():
        assert run_inplace(fn, start) == expected, (key, start)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([[0]], "một ô chết"),
    ([[1]], "một ô sống (chết vì cô đơn)"),
    ([[1, 1]], "hai ô sống cạnh nhau: cả hai chết"),
    ([[1, 1], [1, 1]], "khối 2x2 (still life): không đổi"),
    ([[0, 1, 0], [0, 1, 0], [0, 1, 0]], "blinker dọc: quay thành ngang"),
    ([[0, 0, 0], [1, 1, 1], [0, 0, 0]], "blinker ngang"),
    ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], "toàn chết"),
    ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], "toàn sống 3x3"),
    ([[1], [1], [1]], "một cột ba ô sống"),
    ([[1, 0, 1, 0, 1]], "một hàng thưa"),
    ([[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]], "ví dụ 1 (glider cắt biên)"),
    ([[1] * 25 for _ in range(25)], "25 x 25 toàn sống (kích thước tối đa)"),
]
for start, label in edge_cases:
    truth = oracle(start)
    for key, fn in impls.items():
        assert run_inplace(fn, start) == truth, (key, label, run_inplace(fn, start), truth)
    print("PASS", label)
assert oracle([[1, 1], [1, 1]]) == [[1, 1], [1, 1]]
assert oracle([[0, 1, 0], [0, 1, 0], [0, 1, 0]]) == [[0, 0, 0], [1, 1, 1], [0, 0, 0]]

PASS một ô chết
PASS một ô sống (chết vì cô đơn)
PASS hai ô sống cạnh nhau: cả hai chết
PASS khối 2x2 (still life): không đổi
PASS blinker dọc: quay thành ngang
PASS blinker ngang
PASS toàn chết
PASS toàn sống 3x3
PASS một cột ba ô sống
PASS một hàng thưa
PASS ví dụ 1 (glider cắt biên)
PASS 25 x 25 toàn sống (kích thước tối đa)


### 12.2. Tính chất và phản ví dụ

- **Khối `2×2`** là vật bất biến; **blinker** có chu kỳ `2` (áp dụng hai lần trở về như cũ) — trên bảng đủ rộng để không bị biên cắt.
- Bảng toàn chết luôn toàn chết.
- **Phản ví dụ** cho cập nhật ngây thơ (không đồng thời): trên ví dụ 1 kết quả khác đáp án đúng (xem in ở dưới).
#### Code demo

In [16]:
block = [[0, 0, 0, 0], [0, 1, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]]
assert run_inplace(sol.gameOfLife, block) == block
blinker = [[0, 0, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 1, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 0, 0]]
once = run_inplace(sol.gameOfLife, blinker)
twice = run_inplace(sol.gameOfLife, once)
assert once != blinker and twice == blinker
dead = [[0] * 6 for _ in range(6)]
assert run_inplace(sol.gameOfLife, dead) == dead
print("PASS vật bất biến (khối), chu kỳ 2 (blinker), bảng chết")

naive = copy.deepcopy(examples[0][0])
naive_wrong(naive)
print("phản ví dụ (ví dụ 1): đúng", examples[0][1], "\n| cập nhật ngây thơ:", naive)
assert naive != examples[0][1]

PASS vật bất biến (khối), chu kỳ 2 (blinker), bảng chết
phản ví dụ (ví dụ 1): đúng [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]] 
| cập nhật ngây thơ: [[0, 0, 0], [0, 0, 1], [0, 1, 1], [0, 0, 0]]


### 12.3. Đối chiếu ngẫu nhiên (có seed), nhiều thế hệ và bảng vô hạn
#### Code demo

In [17]:
rng = random.Random(289)
for trial in range(1500):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    p = rng.choice([0.2, 0.4, 0.6])
    board = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]
    truth = oracle(board)
    for key, fn in impls.items():
        assert run_inplace(fn, board) == truth, (key, board)
print("PASS 1500 bảng ngẫu nhiên: mọi cách khớp oracle")

# Nhiều thế hệ liên tiếp: mọi cách phải giữ cùng quỹ đạo với oracle
for trial in range(60):
    m, n = rng.randint(3, 12), rng.randint(3, 12)
    board = [[1 if rng.random() < 0.4 else 0 for _ in range(n)] for _ in range(m)]
    boards = {key: copy.deepcopy(board) for key in impls}
    ref = copy.deepcopy(board)
    for gen in range(12):
        ref = oracle(ref)
        for key, fn in impls.items():
            fn(boards[key])
            assert boards[key] == ref, (key, gen)
print("PASS 60 bảng x 12 thế hệ: quỹ đạo của mọi cách khớp oracle")

# Bảng vô hạn: glider tịnh tiến (+1, +1) sau 4 thế hệ
glider = {(0, 1), (1, 2), (2, 0), (2, 1), (2, 2)}
cur = glider
for _ in range(4):
    cur = step_live_set(cur)
assert cur == {(r + 1, c + 1) for r, c in glider}
print("PASS glider trên bảng vô hạn: sau 4 thế hệ tịnh tiến (+1, +1) và không bị biên cắt")

# Bảng vô hạn so với bảng hữu hạn có đệm lớn: khớp khi ô sống chưa chạm biên
pad = 30
big = [[0] * (2 * pad + 5) for _ in range(2 * pad + 5)]
for r, c in glider:
    big[pad + r][pad + c] = 1
cur = glider
for _ in range(10):
    cur = step_live_set(cur)
    sol.gameOfLife(big)
assert {(r - pad, c - pad) for r in range(len(big)) for c in range(len(big[0])) if big[r][c]} == cur
print("PASS bảng hữu hạn đệm lớn khớp bảng vô hạn trong 10 thế hệ")

PASS 1500 bảng ngẫu nhiên: mọi cách khớp oracle
PASS 60 bảng x 12 thế hệ: quỹ đạo của mọi cách khớp oracle
PASS glider trên bảng vô hạn: sau 4 thế hệ tịnh tiến (+1, +1) và không bị biên cắt


PASS bảng hữu hạn đệm lớn khớp bảng vô hạn trong 10 thế hệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cập nhật không đồng thời (sửa tại chỗ ngây thơ)
Xem `naive_wrong` ở phần kiến thức nền và phần test: kết quả khác đáp án trên ví dụ 1 (với ví dụ 2 thì "may mắn" trùng). **Vì sao sai:** ô duyệt sau đếm hàng xóm dựa trên giá trị **đã đổi** của ô duyệt trước. **Code đúng:** sao chép bảng, hoặc mã hóa trạng thái cũ+mới.

### Lỗi 2: Đếm cả ô đang xét vào số hàng xóm
#### Code sai

In [18]:
def sai_dem_ca_o_giua(board):
    m, n = len(board), len(board[0])
    old = [row[:] for row in board]
    for r in range(m):
        for c in range(n):
            k = 0
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):                               # SAI: gồm cả (0, 0)
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n:
                        k += old[nr][nc]
            board[r][c] = 1 if (k == 3 or (old[r][c] == 1 and k == 2)) else 0


for start in (examples[0][0], examples[1][0]):
    print("đúng:", oracle(start), "| sai:", run_inplace(sai_dem_ca_o_giua, start))

đúng: [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]] | sai: [[0, 1, 0], [1, 0, 0], [1, 0, 1], [0, 1, 0]]
đúng: [[1, 1], [1, 1]] | sai: [[1, 1], [1, 1]]


**Vì sao sai:** ô đang xét **không** là hàng xóm của chính nó. Phải loại `(dr, dc) = (0, 0)`.

### Lỗi 3: Đọc trạng thái đã mã hóa như số nguyên (`board[x][y] == 1` thay vì `in (1, 2)`)
#### Code sai

In [19]:
def sai_doc_ma_hoa(board):
    m, n = len(board), len(board[0])
    for r in range(m):
        for c in range(n):
            k = 0
            for dr, dc in NEIGHBORS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and board[nr][nc] == 1:      # SAI: bỏ sót ô mã 2 (cũ sống, sẽ chết)
                    k += 1
            if board[r][c] == 1 and (k < 2 or k > 3):
                board[r][c] = 2
            elif board[r][c] == 0 and k == 3:
                board[r][c] = 3
    for r in range(m):
        for c in range(n):
            board[r][c] %= 2


bad = [[0, 1, 0], [0, 0, 1], [1, 1, 1], [0, 0, 0]]
print("đúng:", oracle(bad), "\n sai :", run_inplace(sai_doc_ma_hoa, bad))

đúng: [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]] 
 sai : [[0, 0, 0], [0, 0, 1], [0, 1, 1], [0, 0, 0]]


**Vì sao sai:** ô mã `2` (đang sống, sẽ chết) vẫn **là hàng xóm sống ở thế hệ cũ**; bỏ qua nó làm đếm thiếu. Tương tự không được tính ô mã `3` (cũ chết). Điều kiện đúng: `board[nr][nc] in (1, 2)` hoặc `board[nr][nc] & 1` (khi dùng mã bit).

### Lỗi 4: Quên bước giải mã (giá trị `2`, `3` còn trong bảng)
#### Code sai

In [20]:
def sai_quen_giai_ma(board):
    m, n = len(board), len(board[0])
    for r in range(m):
        for c in range(n):
            k = sum(1 for dr, dc in NEIGHBORS
                    if 0 <= r + dr < m and 0 <= c + dc < n and board[r + dr][c + dc] in (1, 2))
            if board[r][c] == 1 and (k < 2 or k > 3):
                board[r][c] = 2
            elif board[r][c] == 0 and k == 3:
                board[r][c] = 3
    # SAI: thiếu lượt giải mã (2 -> 0, 3 -> 1)


print("đúng:", examples[1][1], "| sai:", run_inplace(sai_quen_giai_ma, examples[1][0]))
print("trên ví dụ 1:", run_inplace(sai_quen_giai_ma, examples[0][0]))

đúng: [[1, 1], [1, 1]] | sai: [[1, 1], [1, 3]]
trên ví dụ 1: [[0, 2, 0], [3, 0, 1], [2, 1, 1], [0, 3, 0]]


**Vì sao sai:** bảng chỉ được chứa `0`/`1`. Phải giải mã: `board[r][c] %= 2` (hoặc `>>= 1` với mã bit).

### Lỗi 5: Sao chép nông (alias) `old = board`
#### Code sai

In [21]:
def sai_alias(board):
    m, n = len(board), len(board[0])
    old = board                                                     # SAI: old và board là CÙNG một đối tượng
    for r in range(m):
        for c in range(n):
            k = sum(old[r + dr][c + dc] for dr, dc in NEIGHBORS if 0 <= r + dr < m and 0 <= c + dc < n)
            board[r][c] = 1 if (k == 3 or (old[r][c] == 1 and k == 2)) else 0


print("đúng:", examples[0][1], "\n| alias:", run_inplace(sai_alias, examples[0][0]))

đúng: [[0, 0, 0], [1, 0, 1], [0, 1, 1], [0, 1, 0]] 
| alias: [[0, 0, 0], [0, 0, 1], [0, 1, 1], [0, 0, 0]]


**Vì sao sai:** `old = board` không tạo bản sao; hai tên cùng trỏ một đối tượng. **Code đúng:** `old = [row[:] for row in board]` (sao chép từng hàng) hoặc `copy.deepcopy(board)`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**cập nhật đồng thời mọi ô dựa trên trạng thái cũ**" (Game of Life, mô phỏng khuếch tán, Minesweeper) → **sao chép** hoặc **mã hóa trạng thái cũ+mới** trong cùng ô.
- **Mã hóa `2/3`**: `1 → 2` (sống→chết), `0 → 3` (chết→sống); đọc trạng thái cũ bằng `in (1, 2)`; giải mã `% 2`.
- **Mã hóa bit**: bit `0` = cũ, bit `1` = mới; đọc cũ bằng `& 1`; giải mã `>> 1`.
- Duyệt **8 hướng** bằng hằng `NEIGHBORS`; **kiểm tra biên** trước khi đọc.
- Luật gọn: `sống_mới = (k == 3) or (đang_sống and k == 2)`.
- Bảng vô hạn / thưa → **tập ô sống + `Counter`**; công việc theo số ô sống.
- Kiểm tra bằng các mẫu quen thuộc: khối `2×2` (bất biến), blinker (chu kỳ `2`), glider (tịnh tiến).

## 15. Biến thể của bài

1. **Game of Life vô hạn** (follow-up): Cách 4.
2. **Minesweeper** (529), **Set Matrix Zeroes** (73): kỹ thuật mã hóa trạng thái tại chỗ tương tự.
3. **Rotate Image** (48): tại chỗ với phép hoán vị.
4. **Nhiều thế hệ:** tính thế hệ thứ `g`, phát hiện **chu kỳ** (lưu băm của bảng).
5. **Luật khác** (HighLife `B36/S23`, Seeds `B2/S`): chỉ đổi điều kiện sinh/sống.
#### Code demo

In [22]:
def life_generations(board, g):
    """Tính g thế hệ trên bảng hữu hạn (ngoài biên chết)."""
    b = copy.deepcopy(board)
    for _ in range(g):
        sol.gameOfLife(b)
    return b


def find_cycle(board, limit=200):
    """Trả (chỉ số bắt đầu chu kỳ, độ dài chu kỳ) của chuỗi thế hệ trên bảng hữu hạn."""
    seen = {}
    b = copy.deepcopy(board)
    for gen in range(limit):
        key = tuple(map(tuple, b))
        if key in seen:
            return seen[key], gen - seen[key]
        seen[key] = gen
        sol.gameOfLife(b)
    return None


def step_rule(live, birth, survive):
    counts = Counter((r + dr, c + dc) for r, c in live for dr, dc in NEIGHBORS)
    return {cell for cell, k in counts.items() if (cell in live and k in survive) or (cell not in live and k in birth)}


print(find_cycle(blinker))                                    # (0, 2): chu kỳ 2 ngay từ đầu
print(sorted(step_rule({(0, 0), (0, 1), (1, 0)}, {3}, {2, 3})) == sorted(step_live_set({(0, 0), (0, 1), (1, 0)})))   # True: luật Conway

(0, 2)
True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Sao chép bảng | đọc từ bản sao, ghi vào bảng | `O(N)` | `O(N)` | đơn giản, chắc chắn đúng |
| 2. Mã hóa `2`/`3` | trạng thái trung gian trong ô | `O(N)` | `O(1)` | **cách nộp chuẩn** |
| 3. Mã hóa bit | bit0 cũ, bit1 mới | `O(N)` | `O(1)` | gọn, dễ giải thích |
| 4. Tập ô sống | `Counter` hàng xóm của ô sống | `O(L)` | `O(L)` | bảng thưa/vô hạn (follow-up 2) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `next_state(1, 4)`; (b) `next_state(0, 3)`; (c) `3 % 2`, `2 % 2`; (d) kết quả của khối `2×2` sau một thế hệ.

**Bài 2 (tính bằng tay).** Tính thế hệ kế tiếp của `[[0,1,0],[0,1,0],[0,1,0]]` (blinker dọc) và ghi `k` của từng ô.

**Bài 3 (điền TODO).** Hoàn thiện `life_todo(board)` theo mã hóa bit.
#### Code demo

In [23]:
def life_todo(board):
    m, n = len(board), len(board[0])
    # TODO 1: với mỗi ô (r, c): k = tổng (board[nr][nc] & 1) trên 8 hàng xóm trong biên
    # TODO 2: old = board[r][c] & 1; new = 1 nếu k == 3 hoặc (old == 1 và k == 2), ngược lại 0
    # TODO 3: board[r][c] |= new << 1
    # TODO 4: duyệt lại: board[r][c] >>= 1
    pass


# Tự làm xong rồi mới mở test:
# for start, expected in examples:
#     assert run_inplace(life_todo, start) == expected

**Bài 4 (tự viết lại).** Viết `game_of_life_torus(board)` cho bảng **hình xuyến** (biên "cuộn vòng": hàng `-1` là hàng cuối, cột `-1` là cột cuối, dùng `% m`, `% n`) và kiểm tra rằng glider trên bảng `8 × 8` **quay về** vị trí ban đầu sau `32` thế hệ.

**Bài 5 (biến thể / mini challenge).** Viết `generations_until_stable(board, limit)` trả số thế hệ để bảng hữu hạn trở thành **bất biến** hoặc `None` nếu vượt `limit`; thử với khối `2×2` (`0`), blinker (`None` vì dao động), `[[1]]` (`1`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: luật gọn `k == 3 or (sống and k == 2)`. Bài 2: ô giữa có 2 hàng xóm sống ở trên/dưới. Bài 4: chỉ số `(r + dr) % m`.
- **Mức 2 (vừa):** Bài 1: `0`, `1`, `1` và `0`, khối `2×2` không đổi. Bài 2: ô giữa `(1,1)`: `k = 2` sống; `(1,0)` và `(1,2)` có `k = 3` sinh sản; `(0,1)` và `(2,1)` có `k = 1` chết ⇒ `[[0,0,0],[1,1,1],[0,0,0]]`. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 4: glider tịnh tiến `(+1, +1)` mỗi `4` thế hệ; trên xuyến `8 × 8` cần `4 · 8 = 32` thế hệ để về chỗ cũ. Bài 5: so sánh bảng sau mỗi bước với bảng trước, hoặc dùng `find_cycle` rồi kiểm tra độ dài chu kỳ `== 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert next_state(1, 4) == 0 and next_state(0, 3) == 1 and (3 % 2, 2 % 2) == (1, 0)
assert run_inplace(sol.gameOfLife, [[1, 1], [1, 1]]) == [[1, 1], [1, 1]]

# Bài 2
assert run_inplace(sol.gameOfLife, [[0, 1, 0], [0, 1, 0], [0, 1, 0]]) == [[0, 0, 0], [1, 1, 1], [0, 0, 0]]


# Bài 3
def life_todo_ans(board):
    m, n = len(board), len(board[0])
    for r in range(m):
        for c in range(n):
            k = sum(board[r + dr][c + dc] & 1 for dr, dc in NEIGHBORS if 0 <= r + dr < m and 0 <= c + dc < n)
            old = board[r][c] & 1
            new = 1 if (k == 3 or (old == 1 and k == 2)) else 0
            board[r][c] |= new << 1
    for r in range(m):
        for c in range(n):
            board[r][c] >>= 1


for start, expected in examples:
    assert run_inplace(life_todo_ans, start) == expected


# Bài 4
def game_of_life_torus(board):
    m, n = len(board), len(board[0])
    old = [row[:] for row in board]
    for r in range(m):
        for c in range(n):
            k = sum(old[(r + dr) % m][(c + dc) % n] for dr, dc in NEIGHBORS)
            board[r][c] = 1 if (k == 3 or (old[r][c] == 1 and k == 2)) else 0


torus = [[0] * 8 for _ in range(8)]
for r, c in glider:
    torus[r][c] = 1
start_torus = copy.deepcopy(torus)
for _ in range(32):
    game_of_life_torus(torus)
assert torus == start_torus


# Bài 5
def generations_until_stable(board, limit):
    b = copy.deepcopy(board)
    for gen in range(limit + 1):
        nxt = oracle(b)
        if nxt == b:
            return gen
        b = nxt
    return None


assert generations_until_stable([[1, 1], [1, 1]], 10) == 0
assert generations_until_stable(blinker, 10) is None
assert generations_until_stable([[1]], 10) == 1
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mã hóa `2` = sống→chết, `3` = chết→sống)

```python
class Solution:
    def gameOfLife(self, board: List[List[int]]) -> None:
        m, n = len(board), len(board[0])
        dirs = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
        for i in range(m):
            for j in range(n):
                live = sum(1 for a, b in dirs
                           if 0 <= i + a < m and 0 <= j + b < n and board[i + a][j + b] in (1, 2))
                if board[i][j] == 1 and (live < 2 or live > 3):
                    board[i][j] = 2
                elif board[i][j] == 0 and live == 3:
                    board[i][j] = 3
        for i in range(m):
            for j in range(n):
                board[i][j] %= 2
```

**Checklist**

- [ ] Cập nhật **đồng thời**: đếm hàng xóm chỉ dựa trên **trạng thái cũ**?
- [ ] 8 hướng (loại `(0, 0)`), kiểm tra **biên**?
- [ ] Đọc "cũ sống" bằng `in (1, 2)` (mã `2/3`) hoặc `& 1` (mã bit)?
- [ ] Có lượt **giải mã** cuối (`% 2` hoặc `>> 1`)?
- [ ] Không dùng alias `old = board`; nếu sao chép thì sao chép từng hàng?
- [ ] Nhớ follow-up: bảng vô hạn → tập ô sống + `Counter`?
- [ ] Đã thử khối `2×2`, blinker, bảng toàn chết/toàn sống, `25 × 25`?
- [ ] Skeleton đúng `class Solution` / `def gameOfLife(self, board: List[List[int]]) -> None`?